# Tips 13：Word報告書を自動作成

**業務の困りごと**：毎月同じ体裁のWord報告書を作る。

**このTipsでできること**：`Document` にタイトル・本文・表を追加し、元データを表の行へ書き込んで `.docx` として保存します。

**使用ライブラリ**：python-docx

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

商品と売上金額の2行をWord報告書の元データとして使います。

最後の `display()` で練習用データ全体を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
%pip -q install python-docx
from pathlib import Path
from docx import Document
base=Path('/content') if Path('/content').exists() else Path.cwd()
# Wordの表に入れる商品と金額
rows=[('茶',1200),('菓子',900)]
import pandas as pd
print('報告書に使う元データ（先頭5行）:')
display(pd.DataFrame(rows,columns=['商品','売上金額']))


## 2. 自動処理

`Document` にタイトル・本文・表を追加し、元データを表の行へ書き込んで `.docx` として保存します。

**結果を見るポイント**：Wordの表は見出し1行とデータ2行で、合計は **2,100円** です。

保存した成果物を読み直して `display()` で表示します。画面の表とダウンロードしたファイルの内容を照らし合わせてください。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
# 空のWord文書を作ってタイトルを入れる
doc = Document()
doc.add_heading('月次売上報告', 0)
doc.add_paragraph('2026年9月の集計結果です。')
# 見出し付きの表を作る
table=doc.add_table(rows=1,cols=2)
table.rows[0].cells[0].text='商品'; table.rows[0].cells[1].text='売上金額'
# 商品ごとの行を追加する
for name,value in rows:
    cells = table.add_row().cells
    cells[0].text = name
    cells[1].text = f'{value:,} 円'
doc.add_paragraph(f'合計: {sum(v for _,v in rows):,} 円')
# Wordファイルを保存する
out=base/'Tips13_月次報告.docx'; doc.save(out)
print(out)

# 保存したWordの表の内容を読み直して表示する
saved_table = Document(out).tables[0]
display(pd.DataFrame([[c.text for c in row.cells] for row in saved_table.rows[1:]],
                     columns=[c.text for c in saved_table.rows[0].cells]))


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

Wordの表は見出し1行とデータ2行で、合計は **2,100円** です。


In [ ]:
# 件数や合計が期待値と一致するか検査する
from docx import Document
assert len(Document(out).tables[0].rows)==3
print('確認OK')


## 4. 持ち帰り・練習

**練習問題**：担当者名と作成日を追記しましょう。



**実務で使う前に**：見た目を変えたい場合はWordのスタイルや表の書式を追加できます。

出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
